In [1]:
import sys
import subprocess

# Install torch and torchvision (PyTorch + common vision utilities)
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "torch", "torchvision", "torchaudio"
])

0

In [ ]:
# Install (optional) — safe in normal Jupyter environments.
import sys
import subprocess
from pathlib import Path

# Make `src` importable regardless of notebook working directory.
repo_root = Path.cwd()
if not (repo_root / "src").exists() and (repo_root.parent / "src").exists():
    repo_root = repo_root.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "optuna",
        "optuna-integration[xgboost]",
    ]
)

import os

import numpy as np
import pandas as pd

import optuna
from joblib import dump
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss
from xgboost import XGBClassifier

from src.features import encode_matchups_symmetric
from src.paths import get_data_dir

optuna.logging.set_verbosity(optuna.logging.WARNING)

DATA_DIR = Path(get_data_dir())
MODELS_DIR = Path("models")
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("Using DATA_DIR:", str(DATA_DIR))
print("Saving studies/models to:", str(MODELS_DIR))

# Ensure feature_store exists for downstream cells.
if "feature_store" not in globals():
    print("feature_store not found — building from CSVs...")

    from src.features import compute_efficiency, compute_four_factors
    from src.massey import load_massey_features
    from src.elo import compute_elo_men, compute_elo_women

    # Elo (men + women) into one dict, keyed by (Season, TeamID)
    elo_m = compute_elo_men(str(DATA_DIR))
    elo_w = compute_elo_women(str(DATA_DIR))
    elo_all = dict(elo_m)
    elo_all.update(elo_w)

    # Efficiency / Four Factors (regular season detailed)
    m_det = pd.read_csv(DATA_DIR / "MRegularSeasonDetailedResults.csv")
    w_det = pd.read_csv(DATA_DIR / "WRegularSeasonDetailedResults.csv")
    efficiency = pd.concat([compute_efficiency(m_det), compute_efficiency(w_det)], ignore_index=True)
    four_factors = pd.concat([compute_four_factors(m_det), compute_four_factors(w_det)], ignore_index=True)

    # Massey consensus
    massey = load_massey_features(str(DATA_DIR), min_coverage=0.8)

    # Seeds
    def _seed_num(seed_str: str) -> int | None:
        if not isinstance(seed_str, str):
            return None
        digits = "".join([c for c in seed_str if c.isdigit()])
        if not digits:
            return None
        return int(digits[:2]) if len(digits) >= 2 else int(digits)

    seeds_m_raw = pd.read_csv(DATA_DIR / "MNCAATourneySeeds.csv")[["Season", "TeamID", "Seed"]].copy()
    seeds_w_raw = pd.read_csv(DATA_DIR / "WNCAATourneySeeds.csv")[["Season", "TeamID", "Seed"]].copy()

    seeds_m = seeds_m_raw.assign(
        Season=seeds_m_raw["Season"].astype(int),
        TeamID=seeds_m_raw["TeamID"].astype(int),
        Seed=seeds_m_raw["Seed"].astype(str).map(_seed_num),
    ).dropna(subset=["Seed"])
    seeds_w = seeds_w_raw.assign(
        Season=seeds_w_raw["Season"].astype(int),
        TeamID=seeds_w_raw["TeamID"].astype(int),
        Seed=seeds_w_raw["Seed"].astype(str).map(_seed_num),
    ).dropna(subset=["Seed"])

    feature_store = {
        "elo": elo_all,
        "efficiency": efficiency,
        "four_factors": four_factors,
        "massey": massey,
        "seeds_m": seeds_m[["Season", "TeamID", "Seed"]].copy(),
        "seeds_w": seeds_w[["Season", "TeamID", "Seed"]].copy(),
    }

    print("feature_store built:", {k: (len(v) if isinstance(v, dict) else v.shape) for k, v in feature_store.items()})
else:
    print("feature_store already exists — reusing it.")


In [ ]:
def _build_matchups_from_tourney_compact(tourney_df: pd.DataFrame) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    season = tourney_df["Season"].astype(int).to_numpy()
    w_team = tourney_df["WTeamID"].astype(int).to_numpy()
    l_team = tourney_df["LTeamID"].astype(int).to_numpy()

    team1 = np.minimum(w_team, l_team)
    team2 = np.maximum(w_team, l_team)
    y = (w_team == team1).astype(int)

    matchups_df = pd.DataFrame({"Season": season, "Team1ID": team1, "Team2ID": team2})

    # Recency weights (same formula used elsewhere in prompts)
    w = np.exp(-0.1 * (2024 - season)).astype(float)
    return matchups_df, y, w


def build_Xy_for_seasons(
    *,
    tourney_compact_df: pd.DataFrame,
    seasons: np.ndarray,
    elo_dict: dict[tuple[int, int], float],
    efficiency_df: pd.DataFrame,
    four_factors_df: pd.DataFrame,
    massey_df: pd.DataFrame,
    seeds_df: pd.DataFrame,
) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    df = tourney_compact_df[tourney_compact_df["Season"].astype(int).isin(seasons)].copy()
    matchups_df, y, w = _build_matchups_from_tourney_compact(df)

    encoded = encode_matchups_symmetric(
        matchups_df=matchups_df,
        elo_dict=elo_dict,
        efficiency_df=efficiency_df,
        four_factors_df=four_factors_df,
        massey_df=massey_df,
        seeds_df=seeds_df,
        label_col=pd.Series(y, name="label"),
    )

    y_out = encoded["label"].astype(int).to_numpy()
    # Drop Season if present (we keep it out of modeling features)
    X_out = encoded.drop(columns=["label"]).copy()
    if "Season" in X_out.columns:
        X_out = X_out.drop(columns=["Season"])
    return X_out, y_out, w.repeat(2)  # symmetric doubles rows


def objective_xgb_factory(
    *,
    sex: str,
    tourney_compact_df: pd.DataFrame,
    seeds_df: pd.DataFrame,
) -> callable:
    # Time-respecting split for tuning:
    # Train: 2010-2020, Tune/Val: 2021-2022
    # Keep 2023-2025 untouched for final evaluation (reported after tuning).
    train_seasons = np.arange(2010, 2021)  # 2010-2020
    val_seasons = np.array([2021, 2022])

    X_train, y_train, w_train = build_Xy_for_seasons(
        tourney_compact_df=tourney_compact_df,
        seasons=train_seasons,
        elo_dict=feature_store["elo"],
        efficiency_df=feature_store["efficiency"],
        four_factors_df=feature_store["four_factors"],
        massey_df=feature_store["massey"],
        seeds_df=seeds_df,
    )

    X_val, y_val, _ = build_Xy_for_seasons(
        tourney_compact_df=tourney_compact_df,
        seasons=val_seasons,
        elo_dict=feature_store["elo"],
        efficiency_df=feature_store["efficiency"],
        four_factors_df=feature_store["four_factors"],
        massey_df=feature_store["massey"],
        seeds_df=seeds_df,
    )

    def objective(trial: optuna.Trial) -> float:
        params = dict(
            max_depth=trial.suggest_int("max_depth", 3, 8),
            learning_rate=trial.suggest_float("learning_rate", 0.005, 0.3, log=True),
            n_estimators=trial.suggest_int("n_estimators", 200, 1500),
            subsample=trial.suggest_float("subsample", 0.5, 1.0),
            colsample_bytree=trial.suggest_float("colsample_bytree", 0.5, 1.0),
            min_child_weight=trial.suggest_int("min_child_weight", 1, 10),
            reg_alpha=trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
            reg_lambda=trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
            eval_metric="logloss",
            random_state=42,
        )

        model = XGBClassifier(**params)
        model.fit(X_train, y_train, sample_weight=w_train)

        p = model.predict_proba(X_val)[:, 1]
        return float(brier_score_loss(y_val, p))

    print(
        f"Prepared XGB objective for {sex}: "
        f"X_train={X_train.shape} (2010-2018), X_val={X_val.shape} (2019-2022)"
    )
    return objective


mn_tourney = pd.read_csv(DATA_DIR / "MNCAATourneyCompactResults.csv")
wn_tourney = pd.read_csv(DATA_DIR / "WNCAATourneyCompactResults.csv")

objective_xgb_men = objective_xgb_factory(
    sex="M",
    tourney_compact_df=mn_tourney,
    seeds_df=feature_store["seeds_m"],
)

objective_xgb_women = objective_xgb_factory(
    sex="W",
    tourney_compact_df=wn_tourney,
    seeds_df=feature_store["seeds_w"],
)

In [ ]:
def compute_elo_with_params(
    *,
    compact_regular_path: Path,
    compact_tourney_path: Path,
    k: float,
    hca: float,
    regression_weight: float,
    elo_init: float = 1500.0,
) -> dict[tuple[int, int], float]:
    reg = pd.read_csv(compact_regular_path)
    tou = pd.read_csv(compact_tourney_path)

    cols = ["Season", "DayNum", "WTeamID", "LTeamID", "WLoc"]
    reg = reg[cols].copy()
    tou = tou[cols].copy()

    all_games = pd.concat([reg, tou], ignore_index=True)
    if all_games.empty:
        return {}

    all_games["Season"] = all_games["Season"].astype(int)
    all_games["DayNum"] = all_games["DayNum"].astype(int)
    all_games["WTeamID"] = all_games["WTeamID"].astype(int)
    all_games["LTeamID"] = all_games["LTeamID"].astype(int)
    all_games["WLoc"] = all_games["WLoc"].astype(str)

    all_games = all_games.sort_values(["Season", "DayNum"], kind="mergesort").reset_index(drop=True)

    prev_weight = 1.0 - float(regression_weight)
    mean_weight = float(regression_weight)

    def expected(r_a: float, r_b: float) -> float:
        return 1.0 / (1.0 + 10.0 ** ((r_b - r_a) / 400.0))

    elo_by_team: dict[int, float] = {}
    season_snap: dict[tuple[int, int], float] = {}

    prev_season: int | None = None
    for row in all_games.itertuples(index=False):
        season = int(row.Season)
        if prev_season is None:
            prev_season = season
        elif season != prev_season:
            # snapshot end of prev_season
            for tid, r in elo_by_team.items():
                season_snap[(prev_season, tid)] = float(r)
            # regress to mean
            elo_by_team = {tid: (prev_weight * r + mean_weight * elo_init) for tid, r in elo_by_team.items()}
            prev_season = season

        w = int(row.WTeamID)
        l = int(row.LTeamID)
        w_elo = float(elo_by_team.get(w, elo_init))
        l_elo = float(elo_by_team.get(l, elo_init))

        w_loc = str(row.WLoc)
        if w_loc == "H":
            w_adj = w_elo + float(hca)
        elif w_loc == "A":
            w_adj = w_elo - float(hca)
        else:
            w_adj = w_elo

        e = expected(w_adj, l_elo)
        elo_by_team[w] = w_elo + float(k) * (1.0 - e)
        elo_by_team[l] = l_elo + float(k) * (0.0 - (1.0 - e))

    if prev_season is not None:
        for tid, r in elo_by_team.items():
            season_snap[(prev_season, tid)] = float(r)

    return season_snap


def objective_elo(trial: optuna.Trial) -> float:
    k = trial.suggest_int("K", 10, 40)
    hca = trial.suggest_int("HCA", 50, 200)
    regression_weight = trial.suggest_float("regression_weight", 0.15, 0.40)

    # Recompute Elo for both genders with the same parameters
    elo_m = compute_elo_with_params(
        compact_regular_path=DATA_DIR / "MRegularSeasonCompactResults.csv",
        compact_tourney_path=DATA_DIR / "MNCAATourneyCompactResults.csv",
        k=k,
        hca=hca,
        regression_weight=regression_weight,
    )
    elo_w = compute_elo_with_params(
        compact_regular_path=DATA_DIR / "WRegularSeasonCompactResults.csv",
        compact_tourney_path=DATA_DIR / "WNCAATourneyCompactResults.csv",
        k=k,
        hca=hca,
        regression_weight=regression_weight,
    )

    # Time-respecting split for Elo tuning objective.
    # Train: 2010-2020, Tune/Val: 2021-2022, (Locked test: 2023-2025 reported later)
    train_seasons = np.arange(2010, 2021)  # 2010-2020
    val_seasons = np.array([2021, 2022])

    # Men
    X_train_m, y_train_m, w_train_m = build_Xy_for_seasons(
        tourney_compact_df=mn_tourney,
        seasons=train_seasons,
        elo_dict=elo_m,
        efficiency_df=feature_store["efficiency"],
        four_factors_df=feature_store["four_factors"],
        massey_df=feature_store["massey"],
        seeds_df=feature_store["seeds_m"],
    )
    X_val_m, y_val_m, _ = build_Xy_for_seasons(
        tourney_compact_df=mn_tourney,
        seasons=val_seasons,
        elo_dict=elo_m,
        efficiency_df=feature_store["efficiency"],
        four_factors_df=feature_store["four_factors"],
        massey_df=feature_store["massey"],
        seeds_df=feature_store["seeds_m"],
    )

    # Women
    X_train_w, y_train_w, w_train_w = build_Xy_for_seasons(
        tourney_compact_df=wn_tourney,
        seasons=train_seasons,
        elo_dict=elo_w,
        efficiency_df=feature_store["efficiency"],
        four_factors_df=feature_store["four_factors"],
        massey_df=feature_store["massey"],
        seeds_df=feature_store["seeds_w"],
    )
    X_val_w, y_val_w, _ = build_Xy_for_seasons(
        tourney_compact_df=wn_tourney,
        seasons=val_seasons,
        elo_dict=elo_w,
        efficiency_df=feature_store["efficiency"],
        four_factors_df=feature_store["four_factors"],
        massey_df=feature_store["massey"],
        seeds_df=feature_store["seeds_w"],
    )

    # Quick, stable evaluator: logistic regression on [elo_diff, seed_diff]
    cols = ["elo_diff", "seed_diff"]
    for c in cols:
        if c not in X_train_m.columns:
            raise ValueError(f"Missing feature column: {c}")

    lr_m = LogisticRegression(max_iter=5000)
    lr_m.fit(X_train_m[cols], y_train_m, sample_weight=w_train_m)
    p_m = lr_m.predict_proba(X_val_m[cols])[:, 1]

    lr_w = LogisticRegression(max_iter=5000)
    lr_w.fit(X_train_w[cols], y_train_w, sample_weight=w_train_w)
    p_w = lr_w.predict_proba(X_val_w[cols])[:, 1]

    brier_m = float(brier_score_loss(y_val_m, p_m))
    brier_w = float(brier_score_loss(y_val_w, p_w))

    # Single metric for the study: mean of men/women Brier on 2019-2022
    return (brier_m + brier_w) / 2.0


print("Prepared Elo tuning objective (evaluates mean Brier on 2019-2022 for men+women).")

In [ ]:
# Men XGB (150 trials)
study_men_xgb = optuna.create_study(direction="minimize", study_name="xgb_men")
study_men_xgb.optimize(objective_xgb_men, n_trials=150)
print("Men XGB best_params:", study_men_xgb.best_params)
print("Men XGB best_value (Brier):", study_men_xgb.best_value)
dump(study_men_xgb, MODELS_DIR / "optuna_study_xgb_men.pkl")

# Women XGB (100 trials)
study_women_xgb = optuna.create_study(direction="minimize", study_name="xgb_women")
study_women_xgb.optimize(objective_xgb_women, n_trials=100)
print("Women XGB best_params:", study_women_xgb.best_params)
print("Women XGB best_value (Brier):", study_women_xgb.best_value)
dump(study_women_xgb, MODELS_DIR / "optuna_study_xgb_women.pkl")

# Elo tuning (50 trials)
study_elo = optuna.create_study(direction="minimize", study_name="elo_params")
study_elo.optimize(objective_elo, n_trials=50)
print("Elo best_params:", study_elo.best_params)
print("Elo best_value (Brier):", study_elo.best_value)
dump(study_elo, MODELS_DIR / "optuna_study_elo.pkl")

In [ ]:
import matplotlib.pyplot as plt
import optuna.visualization.matplotlib as ovm
from pathlib import Path

PLOTS_DIR = Path("eda_plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

studies = {
    "xgb_men": study_men_xgb,
    "xgb_women": study_women_xgb,
    "elo": study_elo,
}

for name, study in studies.items():
    ax1 = ovm.plot_optimization_history(study)
    fig1 = ax1.figure
    fig1.set_size_inches(12, 5)
    ax1.set_title(f"Optuna optimization history: {name}")
    fig1.tight_layout()
    out1 = PLOTS_DIR / f"tuning_{name}_history.png"
    fig1.savefig(out1, dpi=150)
    plt.show()
    print("Saved:", out1)

    ax2 = ovm.plot_param_importances(study)
    fig2 = ax2.figure
    fig2.set_size_inches(12, 5)
    ax2.set_title(f"Optuna parameter importances: {name}")
    fig2.tight_layout()
    out2 = PLOTS_DIR / f"tuning_{name}_param_importances.png"
    fig2.savefig(out2, dpi=150)
    plt.show()
    print("Saved:", out2)

In [ ]:
train_seasons = np.arange(2010, 2021)  # 2010-2020
val_seasons = np.array([2021, 2022])
test_seasons = np.array([2023, 2024, 2025])

# Best Elo params (recompute Elo dicts)
bp = study_elo.best_params
elo_best_m = compute_elo_with_params(
    compact_regular_path=DATA_DIR / "MRegularSeasonCompactResults.csv",
    compact_tourney_path=DATA_DIR / "MNCAATourneyCompactResults.csv",
    k=bp["K"],
    hca=bp["HCA"],
    regression_weight=bp["regression_weight"],
)
elo_best_w = compute_elo_with_params(
    compact_regular_path=DATA_DIR / "WRegularSeasonCompactResults.csv",
    compact_tourney_path=DATA_DIR / "WNCAATourneyCompactResults.csv",
    k=bp["K"],
    hca=bp["HCA"],
    regression_weight=bp["regression_weight"],
)

# Build train/val with best Elo
X_tr_m, y_tr_m, w_tr_m = build_Xy_for_seasons(
    tourney_compact_df=mn_tourney,
    seasons=train_seasons,
    elo_dict=elo_best_m,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)
X_va_m, y_va_m, _ = build_Xy_for_seasons(
    tourney_compact_df=mn_tourney,
    seasons=val_seasons,
    elo_dict=elo_best_m,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)

X_tr_w, y_tr_w, w_tr_w = build_Xy_for_seasons(
    tourney_compact_df=wn_tourney,
    seasons=train_seasons,
    elo_dict=elo_best_w,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_w"],
)
X_va_w, y_va_w, _ = build_Xy_for_seasons(
    tourney_compact_df=wn_tourney,
    seasons=val_seasons,
    elo_dict=elo_best_w,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_w"],
)

# Rebuild best XGB models
p_m = dict(study_men_xgb.best_params)
p_m.update({"eval_metric": "logloss", "random_state": 42})
best_model_men = XGBClassifier(**p_m)
best_model_men.fit(X_tr_m, y_tr_m, sample_weight=w_tr_m)

p_w = dict(study_women_xgb.best_params)
p_w.update({"eval_metric": "logloss", "random_state": 42})
best_model_women = XGBClassifier(**p_w)
best_model_women.fit(X_tr_w, y_tr_w, sample_weight=w_tr_w)

# Tune/Val Brier (2021-2022)
p_m_val = best_model_men.predict_proba(X_va_m)[:, 1]
p_w_val = best_model_women.predict_proba(X_va_w)[:, 1]
brier_m_val = float(brier_score_loss(y_va_m, p_m_val))
brier_w_val = float(brier_score_loss(y_va_w, p_w_val))

# Locked Test Brier (2023-2025)
X_te_m, y_te_m, _ = build_Xy_for_seasons(
    tourney_compact_df=mn_tourney,
    seasons=test_seasons,
    elo_dict=elo_best_m,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)
X_te_w, y_te_w, _ = build_Xy_for_seasons(
    tourney_compact_df=wn_tourney,
    seasons=test_seasons,
    elo_dict=elo_best_w,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_w"],
)

p_m_te = best_model_men.predict_proba(X_te_m)[:, 1]
p_w_te = best_model_women.predict_proba(X_te_w)[:, 1]
brier_m_te = float(brier_score_loss(y_te_m, p_m_te))
brier_w_te = float(brier_score_loss(y_te_w, p_w_te))

print("CELL 6 — Rebuild best models")
print("Men Brier (val 2021-2022):", brier_m_val)
print("Women Brier (val 2021-2022):", brier_w_val)
print("Mean Brier (val):", (brier_m_val + brier_w_val) / 2.0)
print("Men Brier (test 2023-2025):", brier_m_te)
print("Women Brier (test 2023-2025):", brier_w_te)
print("Mean Brier (test):", (brier_m_te + brier_w_te) / 2.0)

dump(best_model_men, MODELS_DIR / "best_men.pkl")
dump(best_model_women, MODELS_DIR / "best_women.pkl")
print("Saved:", MODELS_DIR / "best_men.pkl")
print("Saved:", MODELS_DIR / "best_women.pkl")

In [ ]:
import numpy as np

from joblib import dump

from src.nn_blend import train_nn_mlp

print("CELL 6.5 — Train NN models for 3-way blend")

assert "X_tr_m" in globals(), "Expected X_tr_m from CELL 6."
assert "y_tr_m" in globals(), "Expected y_tr_m from CELL 6."
assert "w_tr_m" in globals(), "Expected w_tr_m from CELL 6."
assert "X_tr_w" in globals(), "Expected X_tr_w from CELL 6."
assert "y_tr_w" in globals(), "Expected y_tr_w from CELL 6."
assert "w_tr_w" in globals(), "Expected w_tr_w from CELL 6."

# Train men/women NN on the FULL feature set (same features as XGBoost).
trained_nn_men = train_nn_mlp(
    x=X_tr_m,
    y=y_tr_m,
    sample_weight=w_tr_m,
    random_state=42,
    n_epochs=200,
    patience=20,
)
trained_nn_women = train_nn_mlp(
    x=X_tr_w,
    y=y_tr_w,
    sample_weight=w_tr_w,
    random_state=42,
    n_epochs=200,
    patience=20,
)

# Optionally persist NN models for later inference/blending.
path_nn_m = MODELS_DIR / "best_nn_men.pkl"
path_nn_w = MODELS_DIR / "best_nn_women.pkl"
dump(trained_nn_men, path_nn_m)
dump(trained_nn_women, path_nn_w)
print("Saved:", path_nn_m)
print("Saved:", path_nn_w)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from joblib import dump
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss

from src.calibration import TournamentCalibrator
from src.nn_blend import predict_nn_mlp_proba, tune_blend_weights_lr_xgb_nn

PLOTS_DIR = Path("eda_plots")
PLOTS_DIR.mkdir(parents=True, exist_ok=True)

print("CELL 7 — 3-way blend tuning (LR + XGB + NN) + calibrators")

assert "trained_nn_men" in globals(), "Expected trained_nn_men from CELL 6.5."
assert "trained_nn_women" in globals(), "Expected trained_nn_women from CELL 6.5."
assert "X_tr_m" in globals() and "y_tr_m" in globals() and "w_tr_m" in globals(), "Expected train split from CELL 6."
assert "X_va_m" in globals() and "y_va_m" in globals(), "Expected validation split from CELL 6."

cols_lr = ["elo_diff", "seed_diff"]

# Train LR on the same feature subset used previously.
lr_m = LogisticRegression(max_iter=5000)
lr_m.fit(X_tr_m[cols_lr], y_tr_m, sample_weight=w_tr_m)
lr_w = LogisticRegression(max_iter=5000)
lr_w.fit(X_tr_w[cols_lr], y_tr_w, sample_weight=w_tr_w)

# Validation raw probabilities for 3-way blend.
lr_m_val = lr_m.predict_proba(X_va_m[cols_lr])[:, 1]
lr_w_val = lr_w.predict_proba(X_va_w[cols_lr])[:, 1]

xgb_m_val = best_model_men.predict_proba(X_va_m)[:, 1]
xgb_w_val = best_model_women.predict_proba(X_va_w)[:, 1]

nn_m_val = predict_nn_mlp_proba(trained_nn_men, X_va_m)
nn_w_val = predict_nn_mlp_proba(trained_nn_women, X_va_w)

# Tune 3-way blend weights on 2021-2022.
best = tune_blend_weights_lr_xgb_nn(
    y_men=y_va_m,
    y_women=y_va_w,
    lr_prob_men=lr_m_val,
    xgb_prob_men=xgb_m_val,
    nn_prob_men=nn_m_val,
    lr_prob_women=lr_w_val,
    xgb_prob_women=xgb_w_val,
    nn_prob_women=nn_w_val,
    step=0.1,
)

w_lr = float(best["w_lr"])
w_xgb = float(best["w_xgb"])
w_nn = float(best["w_nn"])
grid = best["grid"]

print("Best weights:")
print("  w_lr:", w_lr)
print("  w_xgb:", w_xgb)
print("  w_nn:", w_nn)
print("  brier_mean_val:", float(best["brier_mean"]))

# Save a quick diagnostic plot from the full grid.
plt.figure(figsize=(10, 5))
sc = plt.scatter(grid["w_lr"], grid["brier_mean"], c=grid["w_nn"], cmap="viridis", s=20, alpha=0.9)
plt.colorbar(sc, label="w_nn")
plt.title("Brier mean on 2021-2022 (3-way blend grid)")
plt.xlabel("w_lr")
plt.ylabel("brier_mean")
plt.tight_layout()
out_path = PLOTS_DIR / "tuning_cell7_three_way_blend_brier_scatter.png"
plt.savefig(out_path, dpi=150)
plt.show()
print("Saved:", out_path)

# Calibrate the FULL blended raw probabilities on 2021-2022.
cal_seasons = np.array([2021, 2022])

# Calibration sets with BEST Elo (so features match the tuned pipeline)
X_cal_m, y_cal_m, _ = build_Xy_for_seasons(
    tourney_compact_df=mn_tourney,
    seasons=cal_seasons,
    elo_dict=elo_best_m,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_m"],
)
X_cal_w, y_cal_w, _ = build_Xy_for_seasons(
    tourney_compact_df=wn_tourney,
    seasons=cal_seasons,
    elo_dict=elo_best_w,
    efficiency_df=feature_store["efficiency"],
    four_factors_df=feature_store["four_factors"],
    massey_df=feature_store["massey"],
    seeds_df=feature_store["seeds_w"],
)

lr_m_cal = lr_m.predict_proba(X_cal_m[cols_lr])[:, 1]
lr_w_cal = lr_w.predict_proba(X_cal_w[cols_lr])[:, 1]

xgb_m_cal = best_model_men.predict_proba(X_cal_m)[:, 1]
xgb_w_cal = best_model_women.predict_proba(X_cal_w)[:, 1]

nn_m_cal = predict_nn_mlp_proba(trained_nn_men, X_cal_m)
nn_w_cal = predict_nn_mlp_proba(trained_nn_women, X_cal_w)

raw_cal_m_blend = np.clip(w_lr * lr_m_cal + w_xgb * xgb_m_cal + w_nn * nn_m_cal, 0.0, 1.0)
raw_cal_w_blend = np.clip(w_lr * lr_w_cal + w_xgb * xgb_w_cal + w_nn * nn_w_cal, 0.0, 1.0)

best_calibrator_men = TournamentCalibrator().fit_men(raw_cal_m_blend, y_cal_m)
best_calibrator_women = TournamentCalibrator().fit_women(raw_cal_w_blend, y_cal_w)

calibrated_m = best_calibrator_men.transform_men(raw_cal_m_blend)
calibrated_w = best_calibrator_women.transform_women(raw_cal_w_blend)

print(
    "Men calibration Brier (blend raw -> calibrated):",
    float(brier_score_loss(y_cal_m, raw_cal_m_blend)),
    "->",
    float(brier_score_loss(y_cal_m, calibrated_m)),
)
print(
    "Women calibration Brier (blend raw -> calibrated):",
    float(brier_score_loss(y_cal_w, raw_cal_w_blend)),
    "->",
    float(brier_score_loss(y_cal_w, calibrated_w)),
)

# Save tuned calibrators
path_cal_m = MODELS_DIR / "best_calibrator_men.joblib"
path_cal_w = MODELS_DIR / "best_calibrator_women.joblib"
dump(best_calibrator_men, path_cal_m)
dump(best_calibrator_women, path_cal_w)
print("Saved:", path_cal_m)
print("Saved:", path_cal_w)

path_weights = MODELS_DIR / "best_blend_weights_lr_xgb_nn.joblib"
dump({"w_lr": w_lr, "w_xgb": w_xgb, "w_nn": w_nn, "step": 0.1}, path_weights)
print("Saved:", path_weights)
